In [1]:
import nest_asyncio
nest_asyncio.apply()   # Lets us use asyncio.run() inside Jupyter

import os
import json
import asyncio
import httpx
from openai import OpenAI
from pprint import pprint

In [2]:
import pandas as pd
import os, json, time
from dotenv import load_dotenv
from openai import OpenAI
import textwrap

In [3]:
def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string

        

load_dotenv(r"C:\Users\arunk\OneDrive\Documents\GenAI\LLM\openai_key.env")   # reads .env file in the current directory

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found! "
        "Make sure you have a .env file with: OPENAI_API_KEY=sk-..."
    )

pretty_print("API key loaded successfully.")

MODEL = 'gpt-5-nano'




client = OpenAI(api_key=api_key)
pretty_print("OpenAI client ready.")

API key loaded successfully.
OpenAI client ready.


Tool Call

In [4]:
def get_weather_manual(latitude: float, longitude: float) -> dict:
    """Call the Open-Meteo API to get current weather for given coordinates."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "current_weather": True
    }
    response = httpx.get(url, params=params)
    response.raise_for_status()
    data = response.json()
    weather = data["current_weather"]
    return {
        "temperature_celsius": weather["temperature"],
        "windspeed_kmh": weather["windspeed"],
        "weather_code": weather["weathercode"],
        "is_day": weather["is_day"] == 1
    }


import httpx

#def geocode_city(city: str):
#    url = "https://nominatim.openstreetmap.org/search"
#    params = {"q": city, "format": "json", "limit": 1}
#    headers = {"User-Agent": "mcp-workshop-demo/1.0"}
#    r = httpx.get(url, params=params, headers=headers, timeout=20)
#    r.raise_for_status()
#    data = r.json()
#    if not data:
#        return None
#    return {"latitude": float(data[0]["lat"]), "longitude": float(data[0]["lon"])}

#print(geocode_city("Tokyo"))

# Quick test
result = get_weather_manual(48.8566, 2.3522)  # Paris
print("Direct API call result for Paris:")
pprint(result)

Direct API call result for Paris:
{'is_day': True,
 'temperature_celsius': 21.0,
 'weather_code': 0,
 'windspeed_kmh': 20.3}


In [5]:
# Tool Schema Definition
weather_tool_schema = {
    "type": "function",
    "name": "get_weather",
    "description": "Get the current weather for a location given its latitude and longitude. "
                   "Returns temperature in Celsius, wind speed, and weather conditions.",
    "parameters": {
        "type": "object",
        "properties": {
            "latitude": {
                "type": "number",
                "description": "Latitude of the location (e.g., 48.8566 for Paris)"
            },
            "longitude": {
                "type": "number",
                "description": "Longitude of the location (e.g., 2.3522 for Paris)"
            }
        },
        "required": ["latitude", "longitude"],
        "additionalProperties": False
    },
    "strict": True
}

print("Tool schema defined ✅")
print(json.dumps(weather_tool_schema, indent=2))

Tool schema defined ✅
{
  "type": "function",
  "name": "get_weather",
  "description": "Get the current weather for a location given its latitude and longitude. Returns temperature in Celsius, wind speed, and weather conditions.",
  "parameters": {
    "type": "object",
    "properties": {
      "latitude": {
        "type": "number",
        "description": "Latitude of the location (e.g., 48.8566 for Paris)"
      },
      "longitude": {
        "type": "number",
        "description": "Longitude of the location (e.g., 2.3522 for Paris)"
      }
    },
    "required": [
      "latitude",
      "longitude"
    ],
    "additionalProperties": false
  },
  "strict": true
}


In [6]:
def chat_with_weather_tool(user_message: str) -> str:
    """
    A complete manual tool-use loop with the Responses API:
    1. Send user message + tool definitions
    2. Check if the model wants to call a function
    3. Execute the function locally
    4. Send the result back
    5. Return the final answer
    """

    # ── First call: the model may request a tool call ──
    response = client.responses.create(
        model=MODEL,
        instructions="You are a helpful assistant. Use the get_weather tool when asked about weather.",
        input=user_message,
        tools=[weather_tool_schema],
    )

    while True:
        function_calls = [item for item in response.output if item.type == "function_call"]

        if not function_calls:
            return response.output_text

        tool_outputs = []

        for fc in function_calls:
            arguments = json.loads(fc.arguments)

            print(f"🔧 LLM requested tool: {fc.name}")
            print(f"   Arguments: {arguments}")

            if fc.name == "get_weather":
                tool_result = get_weather_manual(
                    latitude=arguments["latitude"],
                    longitude=arguments["longitude"],
                )
            else:
                tool_result = {"error": f"Unknown tool: {fc.name}"}

            print(f"   Result: {tool_result}")

            tool_outputs.append({
                "type": "function_call_output",
                "call_id": fc.call_id,
                "output": json.dumps(tool_result),
            })

        response = client.responses.create(
            model=MODEL,
            instructions="You are a deadpool, but have been hired as assistant.",
            previous_response_id=response.id,   # key fix
            input=tool_outputs,                 # only tool outputs
            tools=[weather_tool_schema],        # keep tools available for another round
        )
        return response.output_text

    else:
        return response.output_text

# Let's test it!
answer = chat_with_weather_tool("What's the weather like in Tokyo right now?")
print("\n" + "="*60)
print("💬 Final Answer:")
print(answer)

🔧 LLM requested tool: get_weather
   Arguments: {'latitude': 35.6895, 'longitude': 139.6917}
   Result: {'temperature_celsius': 12.6, 'windspeed_kmh': 2.4, 'weather_code': 2, 'is_day': False}

💬 Final Answer:
Tokyo right now: about 12.6°C with a light breeze of 2.4 km/h. It’s nighttime out there.

If you want, I can pull the 24-hour forecast or translate that weather code (2) into plain English—just say the word. And if you’re heading out, a light jacket should do.


MCP


In [7]:
from agents import Agent, Runner, enable_verbose_stdout_logging
from agents.mcp import MCPServerStdio

enable_verbose_stdout_logging()

In [9]:
import subprocess
import sys
from mcp.os.win32.utilities import FallbackProcess

async def _patched_create_windows_process(command, args, env, errlog, cwd):
    """Patched version that replaces Jupyter's fake stderr with a real pipe."""
    popen_obj = subprocess.Popen(
        [command, *args],
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,  # ← always use PIPE, never Jupyter's stderr
        env=env,
        cwd=cwd,
        bufsize=0,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
    return FallbackProcess(popen_obj)

# Monkey-patch the broken function in the mcp library
import mcp.client.stdio as mcp_stdio
import mcp.os.win32.utilities as mcp_win32

mcp_win32.create_windows_process = _patched_create_windows_process
mcp_stdio._create_platform_compatible_process = lambda command, args, env, errlog, cwd: _patched_create_windows_process(command, args, env, errlog, cwd)

In [10]:
async def agents_sdk_demo():
    mcp_server = MCPServerStdio(
        params={
            "command": "python",
            "args": ["weather_server.py"]
        }
    )

    async with mcp_server:
        agent = Agent(
            name="Weather Assistant",
            instructions=(
                "You are a helpful weather assistant. "
                "Use the available tools to answer questions about weather. "
                "Always provide temperatures in both Celsius and Fahrenheit."
            ),
            mcp_servers=[mcp_server],
        )

        print("🤖 Running agent with query...\n")
        result = await Runner.run(
            starting_agent=agent,
            input="What's the weather in New York City right now? "
                  "Also, what do the weather codes mean?"
        )

        print("💬 Agent's Response:")
        print(result.final_output)

await agents_sdk_demo()

🤖 Running agent with query...

Creating trace Agent workflow with id trace_431cd3b5639d47448f0254847f250449
Setting current trace: trace_431cd3b5639d47448f0254847f250449
Creating span <agents.tracing.span_data.MCPListToolsSpanData object at 0x0000026E5CD02D00> with id None
Creating span <agents.tracing.span_data.AgentSpanData object at 0x0000026E388632F0> with id None
Running agent Weather Assistant (turn 1)
No conversation_id available for request
Creating span <agents.tracing.span_data.ResponseSpanData object at 0x0000026E5D020280> with id None
Calling LLM
LLM responded
Processing output item type=function_call class=ResponseFunctionToolCall
Creating span <agents.tracing.span_data.FunctionSpanData object at 0x0000026E5D036510> with id None
Invoking MCP tool get_weather
MCP tool get_weather completed.
Creating span <agents.tracing.span_data.MCPListToolsSpanData object at 0x0000026E5D0968A0> with id None
Running agent Weather Assistant (turn 2)
No conversation_id available for request


Exported 2 items


Two MCP Servers

In [11]:
async def multi_server_demo():
    weather_server = MCPServerStdio(
        params={"command": "python", "args": ["weather_server.py"]}
    )
    notes_server = MCPServerStdio(
        params={"command": "python", "args": ["notes_server.py"]}
    )

    async with weather_server, notes_server:
        agent = Agent(
            name="Personal Assistant",
            instructions=(
                "You are a personal assistant with access to weather data and a notes system. "
                "You can check weather anywhere in the world and manage notes. "
                "When the user asks about weather, check it and offer to save the info as a note."
            ),
            mcp_servers=[weather_server, notes_server],
        )

        print("🚀 Agent has tools from BOTH servers!\n")

        result = await Runner.run(
            starting_agent=agent,
            input=(
                "Check the weather in Tokyo and Paris right now, "
                "then save a note with the comparison results. "
                "Title it 'Weather Check' with tags: weather, travel"
            )
        )

        print("💬 Agent's Response:")
        print(result.final_output)

asyncio.run(multi_server_demo())

🚀 Agent has tools from BOTH servers!

Creating trace Agent workflow with id trace_aea7c106372e4bc3b18404f6f68395c2
Setting current trace: trace_aea7c106372e4bc3b18404f6f68395c2
Creating span <agents.tracing.span_data.MCPListToolsSpanData object at 0x0000026E5D0B28F0> with id None
Creating span <agents.tracing.span_data.MCPListToolsSpanData object at 0x0000026E5D020780> with id None
Creating span <agents.tracing.span_data.AgentSpanData object at 0x0000026E5D034770> with id None
Running agent Personal Assistant (turn 1)
No conversation_id available for request
Creating span <agents.tracing.span_data.ResponseSpanData object at 0x0000026E5D0B3570> with id None
Calling LLM
Exported 3 items
LLM responded
Processing output item type=function_call class=ResponseFunctionToolCall
Creating span <agents.tracing.span_data.FunctionSpanData object at 0x0000026E5D0A4AD0> with id None
Invoking MCP tool get_weather_comparison
MCP tool get_weather_comparison completed.
Creating span <agents.tracing.span_

Exported 6 items


In [12]:
from mcp.client.stdio import StdioServerParameters,stdio_client
from mcp import ClientSession

async def demo_mcp_client():
    """
    Demonstrates the MCP client lifecycle: 
    start a server process, send messages, and receive responses.
    """

    server_params = StdioServerParameters(
        command="python",
        args=["weather_server.py"],
    )

    print("Connecting to MCP server...")

    async with stdio_client(server_params) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()

            print()
            print("✅ MCP session initialized!")

            tools_result = await session.list_tools()

            print(f"Server exposes {len(tools_result.tools)} tool(s):")

            print("Available tools from server:")            
            for tool in tools_result.tools:
                print(f" - {tool.name}: {tool.description}")

                schema_preview = json.dumps(tool.inputSchema, indent=2)
                print(f"   Input preview:\n{schema_preview}\n")


            resources_result = await session.list_resources()
            print(f"Server exposes {len(resources_result.resources)} resource(s):")
            print("Available resources from server:")
            for resource in resources_result.resources:
                res_description = resource.description if resource.description else "No description"
                print(f" - {resource.name}: {res_description}")
                print(f"   URI: {resource.uri} : {res_description[:100]}...\n")  


            #Sample Tool Call in MCP server by MCP client
            result = await session.call_tool(
                "get_weather",
                arguments={
                    "latitude": 40.7128,
                    "longitude": -74.0060
                }
            )
            print("Sample tool call result:")
            print(result.content[0].text)

asyncio.run(demo_mcp_client())

Connecting to MCP server...

✅ MCP session initialized!
Server exposes 2 tool(s):
Available tools from server:
 - get_weather: Get the current weather for a location.

    Args:
        latitude: Latitude of the location (e.g., 48.8566 for Paris)
        longitude: Longitude of the location (e.g., 2.3522 for Paris)

    Returns:
        A JSON string with temperature, wind speed, and conditions.
    
   Input preview:
{
  "properties": {
    "latitude": {
      "title": "Latitude",
      "type": "number"
    },
    "longitude": {
      "title": "Longitude",
      "type": "number"
    }
  },
  "required": [
    "latitude",
    "longitude"
  ],
  "title": "get_weatherArguments",
  "type": "object"
}

 - get_weather_comparison: Compare current weather between two cities.

    Args:
        city1_lat: Latitude of the first city
        city1_lon: Longitude of the first city
        city1_name: Name of the first city
        city2_lat: Latitude of the second city
        city2_lon: Longitud